# 11 深度學習（PyTorch）：用神經網路預測感染

同事問：「要不要試試深度學習？」
我們用 PyTorch 建一個簡單的二元分類網路，看看 280 筆資料上能做到什麼程度。

流程：**資料前處理 → 模型架構 → 訓練迴圈 → 早停法 → AUC 評估 → 學習曲線 → 與 sklearn 比較**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 資料前處理（手動轉 tensor）---
import pathlib

import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 與 Ch10 相同的特徵（不用症狀，避免 data leakage）
num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

# One-hot 編碼類別特徵
X_df = pd.get_dummies(df[num_cols + cat_cols + bin_cols], drop_first=True)
X_np = X_df.values.astype(np.float32)
y_np = df["infected"].values.astype(np.float32)

# 標準化 age
scaler = StandardScaler()
X_np[:, 0] = scaler.fit_transform(X_np[:, 0:1]).ravel()

# 70/30 split
idx = np.arange(len(X_np))
np.random.shuffle(idx)
split = int(0.7 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

X_train = torch.tensor(X_np[train_idx])
y_train = torch.tensor(y_np[train_idx]).unsqueeze(1)
X_val = torch.tensor(X_np[val_idx])
y_val = torch.tensor(y_np[val_idx]).unsqueeze(1)

print(f"特徵維度：{X_train.shape[1]}")
print(f"訓練集：{len(X_train)}，驗證集：{len(X_val)}")
print(f"特徵名稱：{list(X_df.columns)}")

In [ ]:
# --- Step 2: 模型架構 ---
# input_dim \u2192 32 \u2192 16 \u2192 1
input_dim = X_train.shape[1]

model = nn.Sequential(
    nn.Linear(input_dim, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

# 參數數量
n_params = sum(p.numel() for p in model.parameters())
print(f"模型架構：{input_dim} \u2192 32 \u2192 16 \u2192 1")
print(f"總參數量：{n_params}")
print(f"參數 / 樣本比：{n_params / len(X_train):.1f}")
print(f"\n\u2192 參數比樣本還多 \u2192 過擬合風險極高！")

In [ ]:
# --- Step 3: 訓練迴圈 + 早停法 ---
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 記錄歷史
train_losses, val_losses = [], []
best_val_loss = float("inf")
patience, counter = 15, 0
best_state = None
best_epoch = 0

for epoch in range(300):
    # 訓練
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    # 驗證
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = loss_fn(val_logits, y_val).item()
    val_losses.append(val_loss)

    # 早停
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# 載入最佳模型
model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch}, best val_loss: {best_val_loss:.4f}")

In [ ]:
# --- Step 4: 學習曲線 ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train Loss", color="#2c7fb8")
ax.plot(val_losses, label="Val Loss", color="#e34a33")
ax.axvline(x=best_epoch, color="gray", linestyle="--", alpha=0.5,
           label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.set_title("Learning Curve")
ax.legend()
plt.tight_layout()
plt.show()

print("\u2192 如果 train loss 持續下降但 val loss 反彈 \u2192 過擬合")
print("\u2192 早停法在 val loss 不再改善時停止訓練")

In [ ]:
# --- Step 5: AUC 評估 ---
model.eval()
with torch.no_grad():
    val_proba = torch.sigmoid(model(X_val)).numpy()
    train_proba = torch.sigmoid(model(X_train)).numpy()

auc_train = roc_auc_score(y_train.numpy(), train_proba)
auc_val = roc_auc_score(y_val.numpy(), val_proba)

print(f"=== PyTorch DL 結果 ===")
print(f"Train AUC = {auc_train:.3f}")
print(f"Val   AUC = {auc_val:.3f}")
print(f"Gap       = {auc_train - auc_val:.3f}")

if auc_train - auc_val > 0.1:
    print("\n\u2192 Train-Val gap > 0.1 \u2192 過擬合嚴重")
    print("\u2192 280 筆資料不足以支撐這個模型的參數量")
else:
    print("\n\u2192 Gap 不大，模型相對穩定")

In [ ]:
# --- Step 6: 與 sklearn 比較 ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 用相同的 train/val split
X_full = df[num_cols + cat_cols + bin_cols]
y_full = df["infected"]

X_sk_train = X_full.iloc[train_idx]
X_sk_val = X_full.iloc[val_idx]
y_sk_train = y_full.iloc[train_idx]
y_sk_val = y_full.iloc[val_idx]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

compare = []

# Logistic Regression
clf_lr = Pipeline([("pre", preprocess), ("model", LogisticRegression(max_iter=500, random_state=42))])
clf_lr.fit(X_sk_train, y_sk_train)
auc_lr = roc_auc_score(y_sk_val, clf_lr.predict_proba(X_sk_val)[:, 1])
compare.append(("Logistic Regression", auc_lr))

# Random Forest
clf_rf = Pipeline([("pre", preprocess), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
clf_rf.fit(X_sk_train, y_sk_train)
auc_rf = roc_auc_score(y_sk_val, clf_rf.predict_proba(X_sk_val)[:, 1])
compare.append(("Random Forest", auc_rf))

# PyTorch
compare.append(("PyTorch DL", auc_val))

print("=== 模型比較（相同 train/val split）===")
for name, auc in compare:
    print(f"  {name:25s}  Val AUC = {auc:.3f}")

print("\n\u2192 在 280 筆資料上，三個模型的表現通常很接近")
print("\u2192 DL 並未展現明顯優勢，反而有過擬合風險")
print("\u2192 教學價值：學會 PyTorch 語法，未來遇到大資料集才能派上用場")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 前處理 | `pd.get_dummies()` + `torch.tensor()` 手動轉換 |
| 模型 | `nn.Sequential(Linear \u2192 ReLU \u2192 Linear \u2192 ReLU \u2192 Linear)` |
| 訓練迴圈 | `zero_grad \u2192 forward \u2192 loss \u2192 backward \u2192 step` |
| 早停法 | 監控 val_loss，patience 到了就停 |
| 學習曲線 | 視覺化 train/val loss 診斷過擬合 |
| 模型比較 | 相同 split 下公平比較 DL vs sklearn |

**結論**：
- 280 筆 \u2192 DL 過殺，sklearn 就夠了
- 但 PyTorch 語法值得學：未來遇到影像、序列、大樣本就需要
- 重點不是「哪個模型最強」，而是「用正確的工具解決正確的問題」

下一章（Ch12），我們問：淋浴真的「導致」感染嗎？ \u2192 因果推論。